### This script is used to clean the datatables used in the Warhammer database and export parquet files that will be used in the database

In [93]:
import pandas as pd
import numpy as np

## Read in Data
We'll read in the raw data using `pd.read_csv`. Reminder: These data should be obtained from kaggle at the following link [https://www.kaggle.com/datasets/theredmage/warhammer-40k].

For specific download sections see 'Data Access' section of `README.md` in the repository.

In [94]:
# Datasheet table contains the name of the model, the model ID, and the ID faction it belongs to
datasheets_raw = pd.read_csv("data/Wahapedia Data Export - Datasheets.csv")

# Models table contains the model name and the size of models
models_raw = pd.read_csv("data/Wahapedia Data Export - DS_Models.csv")

# Model cost table contains the model id's, and the cost of each
model_cost_raw = pd.read_csv("data/Wahapedia Data Export - DS_Model Costs.csv")

# Factions table contains faction ID's, and associated names
factions_raw = pd.read_csv("data/Wahapedia Data Export - Factions.csv")

## Data Cleaning
We want to select the desired columns from each table so we don't have any data that we don't need. Also, some of the tables have overlapping column names for separate categories - so we're going to rename them to avoid mixups

In [ ]:
# 'id' in datasheets is a model id
datasheets = datasheets_raw[['id', 'name', 'faction_id']].rename(columns={'id': 'model_id'})

# 'datasheet_id' refers to a model_id. Some models have multiples entries or "lines" because there is an alternavite version
# These alternative versions a have redundant id's, but different legends, so we can remove them
models = models_raw[['datasheet_id', 'line', 'name', 'M']].rename(columns={'M': 'size', 'datasheet_id' : 'model_id'})
# Remove any 'line' that isn't '1', this ensures we don't have those duplicates discussed above
models = models[models['line']==1].drop(columns = 'line')

# Similar for model costs, we have reduntant entries that are notated in 'line' column
# We only want to keep line 1's
model_cost = model_cost_raw[model_cost_raw['line'] == 1].drop(columns = 'line')

# 'id' in factions is a faction id
factions = factions_raw[['id', 'name']].rename(columns={'id': 'faction_id', 'name':'faction_name'})



We want to make sure that the datatypes for all of our columns make sense. For anything numeric, we want 'int64'. Anything that has text should be an 'object'.

In [96]:
# Check data types
print(f"datasheet datatypes are: \n{datasheets.dtypes}")
datasheets.head()


datasheet datatypes are: 
model_id       int64
name          object
faction_id    object
dtype: object


,model_id,name,faction_id
0,1,Warboss,ORK
1,2,Warboss In Mega Armour,ORK
2,3,Warboss On Warbike,ORK
3,4,Weirdboy,ORK
4,6,Big Mek In Mega Armour,ORK


We've got the right data types for all of our columns in the 'datasheets' table. Now we want to check unique values to make sure all of our values make sense.

In [97]:
# First, let's check how many rows we have in our table
datasheets.shape

(1636, 3)

We have 1636 rows in our table. This means that there should be 1636 unique models. In Warhammer, some models share names, but they look different and can have different cost depending on what faction they belong to. We want to make sure that there are 1636 unique model_ids, and 1636 unique combinations of 'name' and 'faction_id'

In [98]:

# Check for number of unique model_ids, should be the same as our number of rows
print(f"Unique model_ids: {datasheets['model_id'].nunique()}")

# Now check for unique combination of 'name' and 'faction_id', this should also be 1636
print(f"Unique name/faction_id combos: {datasheets[['name', 'faction_id']].drop_duplicates().shape[0]}")


Unique model_ids: 1636
Unique name/faction_id combos: 1626


Looks like we have 10 duplicates in our name/faction_id columns, let's find out which ones they are, and remove them from our datasheets table.

In [99]:
# Find the model_id's of all the duplicates, sort them so it's easier to read
# We want to save the model_ids that are duplicates so we can remove them from other tables
dupes = datasheets[datasheets.duplicated(subset=['name', 'faction_id'], keep=False)].sort_values(['name', 'faction_id'])
dupes


,model_id,name,faction_id
1092,2705,Gladiator Lancer,SM
1164,2787,Gladiator Lancer,SM
871,1667,Gladiator Reaper,SM
1166,2789,Gladiator Reaper,SM
875,1825,Gladiator Valiant,SM
1165,2788,Gladiator Valiant,SM
1011,2568,Impulsor,SM
1163,2786,Impulsor,SM
638,1104,Karanak,CD
1588,4102,Karanak,CD


In [100]:

# Overwrite our table but drop duplicates
datasheets_clean = datasheets = datasheets.drop_duplicates(subset=['name', 'faction_id'], keep='first')

# Check for number of unique model_ids, should be the same as our number of rows
print(f"Unique model_ids: {datasheets_clean['model_id'].nunique()}")

# Now check for unique combination of 'name' and 'faction_id', this should also be 1636
print(f"Unique name/faction_id combos: {datasheets_clean[['name', 'faction_id']].drop_duplicates().shape[0]}")

datasheets_clean.shape

Unique model_ids: 1626
Unique name/faction_id combos: 1626


(1626, 3)

Nice and clean, lets move on to our **models** table

In [101]:
# First, check out the shape
models.shape

(1635, 3)

Right now our 'size' metric has quotations to indicate the units, the NAN size category is notated by a `-`, and the models larger than 16 inches are called 20+. Eventually we want to re-categorize into small (equal to or less than 8), medium (more than 8, less than or equal and 14) and large (>14). It'll take a couple steps to get there, and then we validate the values that we have. First, we want to check the shape of the table. There should only be one size and name for each model_id, and we should have the same model_id's that we have in our datasheet table

#### 1) Check table shape, match model_id's to the model_ids found in 'datasheets'

In [102]:
models['model_id'].nunique()

1635

In [103]:
# We've got more model_ids than our datasheet_clean table. Let's remove the duplicates
models_nodupes = models[models['model_id'].isin(datasheets_clean['model_id'])]
models_nodupes.shape

(1625, 3)

Now we've got one less, 1625 instead of 1626. Looks like there's one model ID that does not have an entry in the model size table. Let's find out which one

In [104]:
# Filter for model_ids that are in datasheets, but not modules_nodupes
datasheets_clean[datasheets_clean['model_id'].isin(set(datasheets_clean['model_id']) - set(models_nodupes['model_id']))]

,model_id,name,faction_id
1308,3708,Example Wargear,SM


With a little bit of internet research, we find out that 'Example Wargear' isn't even a real model for the Space Marines (SM) faction - so lets just remove this column from datasheets.

In [105]:
# Remove model_id 3708 from datasheets_clean, and overwrite
datasheets_clean = datasheets_clean[datasheets_clean['model_id'] != 3708]

Now we have 1625 unique model Id's in both tables. `datasheets_clean` has information on the faction that each model belongs to, and `models_nodupes` has information on size for each unique model_id. Now let's make sure our size category makes sense. For our analysis, we will eventually want size to be categories 'small', 'medium' and 'large'.

**First**, let's take a look at our data types

In [106]:
# Check data types and unique values for size
print(f"models datatypes are: \n{models.dtypes}")
models['size'].unique()

models datatypes are: 
model_id     int64
name        object
size        object
dtype: object


array(['6"', '5"', '12"', '20+"', '3"', '10"', '8"', '14"', '-', '9"',
       '7"', '4"', '16"', '13"'], dtype=object)

Right now our 'size' metric has quotations to indicate the units, the NAN size category is notated by a `-`, and the models larger than 16 inches are called 20+. Eventually we want to re-categorize into small (equal to or less than 8), medium (more than 8, less than or equal and 14) and large (>14). It'll take a couple steps to get there, and then we validate the values that we have.

In [107]:
# First, eliminate the symbols that are preventing this column from being numeric
models_nodupes['size'] = models_nodupes['size'].str.replace('"', '', regex=False).str.replace('+', '', regex=False).replace('-', pd.NA)
# Make 'size' an integer
models_nodupes['size'] = pd.to_numeric(models_nodupes['size'], errors='coerce').astype('Int64')
models_nodupes.dtypes

/var/folders/sc/q60_7bc9267221rzq5p0j72c0000gn/T/ipykernel_32581/804888545.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  models_nodupes['size'] = models_nodupes['size'].str.replace('"', '', regex=False).str.replace('+', '', regex=False).replace('-', pd.NA)
/var/folders/sc/q60_7bc9267221rzq5p0j72c0000gn/T/ipykernel_32581/804888545.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  models_nodupes['size'] = pd.to_numeric(models_nodupes['size'], errors='coerce').astype('Int64')


model_id     int64
name        object
size         Int64
dtype: object

Now we have a numeric column for size - this will make it easier to categorize. Let's make sure we have unique values in 'size' that make sense, and then categorize them into small, medium and large

In [108]:
models_nodupes['size'].unique()

<IntegerArray>
[6, 5, 12, 20, 3, 10, 8, 14, <NA>, 9, 7, 4, 16, 13]
Length: 14, dtype: Int64

In [110]:
# Define new categories for size column as a function
def size_category(x):
    if pd.isna(x): return pd.NA # Retain NAs
    elif x <= 8:   return 'small' # Less than or equal to 8, small
    elif x <= 13:  return 'medium' # Less than or equal to 13, medium
    else:          return 'large' # Greater than 13, large

# Make a copy of our table to apply the change to
models_clean = models_nodupes.copy()

# Map our size category function to 
models_clean['size'] = models_clean['size'].map(size_category)

models_clean['size'].unique()

array(['small', 'medium', 'large', <NA>], dtype=object)

Our **models** tables is now clean. Let's move on to model_costs

First, let's take a look at our table to see the number of rows, and the column names

In [115]:
print(model_cost.shape)
model_cost.head()

(1632, 3)


,datasheet_id,description,cost
0,1,1 model,75
1,2,1 model,80
2,3,1 model,75
3,4,1 model,65
4,6,1 model,90


The 'datasheet_id' column corresponds to the 'model_id' column in our other tables, and it looks like we have a different number of model_ids that both of our other tables. Let's change the name of datasheet_id to model_id, and then address the number of columns

In [118]:
# Rename column
model_cost = model_cost.rename(columns={'datasheet_id': 'model_id'})

# Drop duplicates of 'model_id'
model_cost = model_cost.drop_duplicates(subset=['model_id'])

# Check number of rows
model_cost.shape

(1632, 3)

Looks like there weren't any duplicates. Let's retain only the model_ids in `model_cost` that also exist in `datasheets`

In [120]:
model_cost_matched = model_cost[model_cost['model_id'].isin(datasheets_clean['model_id'])]
model_cost_matched.shape

(1622, 3)

Looks like we have 4 model_ids that don't have an associated value in model_cost. Since we will eventually be doing a query that investigates the relationships between cost, size and faction of models, we have no use for model_ids that don't have an associated cost. Let's identify the missing model_ids, and remove them from datasheets_clean and models_clean.

In [121]:
# Identify missing model_ids
set(datasheets_clean['model_id']) - set(model_cost_matched['model_id'])

{785, 2770, 2807}

In [146]:
# Remove from datasheets and models, overwrite
valid_ids = set(model_cost_matched['model_id'])
datasheets_clean = datasheets_clean[datasheets_clean['model_id'].isin(valid_ids)]
models_clean = models_clean[models_clean['model_id'].isin(valid_ids)]
# Rename model cost table for consistency
model_cost_clean = model_cost_matched

# Identify the number of model_ids in each of the three tables
print(f"datasheets_clean: {datasheets_clean['model_id'].nunique()}, models_clean: {models_clean['model_id'].nunique()}, model_cost_matched: {model_cost_matched['model_id'].nunique()}")

# Check if they all have the same model_ids with assert
# No output means that all tables have matching model_ids
assert set(datasheets_clean['model_id']) == set(models_clean['model_id']) == set(model_cost_matched['model_id'])


datasheets_clean: 1622, models_clean: 1622, model_cost_matched: 1622


Okay, now we have the same model_ids in all of our tables. Now we can finish cleaning model_costs. Let's check the datatypes

In [131]:
print(f"model costs datatypes are: \n{model_cost_matched.dtypes}")
print(f"range of model_costs_matched are: {model_cost_matched['cost'].min()} to {model_cost_matched['cost'].max()} ")

model costs datatypes are: 
model_id        int64
description    object
cost            int64
dtype: object
range of model_costs_matched are: 20 to 3500 


Looks like we've got a model that costs 3500 dollars. This seems extreme. Let's do a little investigation. Let's find out what this model is so we can google it.

In [137]:
model_cost_matched[model_cost_matched['cost'] == 3500].merge(datasheets_clean[['model_id', 'name']], on='model_id')

,model_id,description,cost,name
0,869,1 model,3500,Warlord Titan


Wow, according to the warhammer website, the body alone of the Warlord Titan is $2,000.  The fully kitted Warlord Titan model costs approximately $3500. Looks like we can rest assured that there are some very expensive models. [https://www.warhammer.com/en-US/shop/mars-pattern-warlord-titan-body?srsltid=AfmBOoo9WdYMk9uyyA2lnUaOsAK1DGuKH0XuKfimd4moL9--cmk1pH85]

Let's move on ot our **factions** table

This table should just have none row for each faction. 'faction_id' is the foreign key, this table is just so we can identify the names of factions. First let's check that we have the same number of rows in this table as we have unique faction_ids in 'datasheets_clean'

In [140]:
# First, let's just take a look
factions.head()

,faction_id,faction_name
0,AC,Adeptus Custodes
1,AdM,Adeptus Mechanicus
2,AE,Aeldari
3,AM,Astra Militarum
4,AoI,Imperial Agents


In [141]:
# Assert that the unique values of faction_id are the same between both tables
# No return means it's true
assert set(factions['faction_id']) == set(datasheets_clean['faction_id'])

Now we know that all of the factions in our factions table are represented in datasheets. Now let's make sure our datatypes makes sense. Both columns should be 'object'

In [142]:
print(f"factions datatypes are: \n{factions.dtypes}")

factions datatypes are: 
faction_id      object
faction_name    object
dtype: object


Looks like our **factions** table didn't need any cleaning! What a miracle.

Now, let's write all of our tables to parquet files, and create our database. First, install `pyarrow`, a tool for making parquet files. Then, we'll write our files, put them into the 'outputs' folder.

In [143]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [147]:
model_cost_clean.to_parquet("outputs/model_cost.parquet", index=False)
factions.to_parquet("outputs/factions.parquet", index=False)
models_clean.to_parquet("outputs/models.parquet", index=False)
datasheets_clean.to_parquet("outputs/datasheets.parquet", index=False)

Now we have a folder full of new, clean parquet files. Let's turn them into a database! Then, we can move to queries.sql and do some exploration!

In [148]:
# Create database
!pip install duckdb
import duckdb

conn = duckdb.connect("wh_database.duckdb")

for parquet_file, table_name in [
    ("outputs/models.parquet", "models"),
    ("outputs/model_cost.parquet", "model_cost"),
    ("outputs/factions.parquet", "factions"),
    ("outputs/datasheets.parquet", "datasheets"),
]:
    conn.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_parquet('{parquet_file}')")

conn.close()